# Advanced EDA & Feature Engineering

This notebook demonstrates a complete, production-ready pipeline for Exploratory Data Analysis and Feature Engineering.

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Add src to path
sys.path.append(os.path.abspath('../'))

from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.visualization import Visualizer
from src.config import RAW_DATA_PATH, OUTPUTS_DIR

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading
Let's load our raw e-commerce dataset.

In [2]:
df = pd.read_csv(RAW_DATA_PATH)
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,ItemsInCart,ReferralSource,CustomerRating,TotalPrice
0,ORD000001,2023-01-01 00:00:00,C25795,Laptop,4.0,206.664611,774 Main St,Online,Shipped,4.0,Referral,3.664780,826.658444
1,ORD000002,2023-01-01 01:00:00,C10860,Tablet,NaN,743.812052,215 Main St,Cash,Delivered,6.0,Google,2.049393,743.812052
2,ORD000003,2023-01-01 02:00:00,C86820,Desk,3.0,899.067407,704 Main St,Credit Card,Pending,NaN,Instagram,1.813224,2697.202220
3,ORD000004,2023-01-01 03:00:00,C64886,Laptop,2.0,834.992910,141 Main St,Online,Delivered,5.0,Referral,3.103828,1669.985821
4,ORD000005,2023-01-01 04:00:00,C16265,Laptop,4.0,174.746628,895 Main St,Gift Card,Shipped,9.0,Facebook,NaN,698.986511


## 2. Initial Exploratory Data Analysis (EDA)
Visualizing missing values, distributions, and outliers before preprocessing.

In [3]:
vis = Visualizer(df, OUTPUTS_DIR)
vis.plot_missing_values()
vis.plot_histograms()
vis.plot_boxplots()
print("Plots have been saved to the 'outputs/' directory.")

2026-07-23 18:56:10,925 - INFO - Missing values plot saved.
2026-07-23 18:56:12,610 - INFO - Histograms saved.
2026-07-23 18:56:13,060 - INFO - Boxplots saved.


Plots have been saved to the 'outputs/' directory.


## 3. Preprocessing (Missing Values & Outliers)
- **<5% Missing:** Dropped
- **5-20% Missing:** Median Imputed
- **>20% Missing:** KNN Imputed
- **Outliers:** Detected using IQR and winsorized.

In [4]:
preprocessor = DataPreprocessor(df)
df_clean = preprocessor.handle_missing_values()
df_clean = preprocessor.detect_and_handle_outliers(method='iqr', action='winsorize')
df_clean.head()

2026-07-23 18:56:13,080 - INFO - Column 'Quantity' has 3.00% missing values.
2026-07-23 18:56:13,082 - INFO - Dropping rows for 'Quantity' (<5% missing).
2026-07-23 18:56:13,088 - INFO - Column 'ItemsInCart' has 15.00% missing values.
2026-07-23 18:56:13,090 - INFO - Applying Median imputation for 'ItemsInCart' (5-20% missing).
2026-07-23 18:56:13,092 - INFO - Column 'CustomerRating' has 25.00% missing values.
2026-07-23 18:56:13,093 - INFO - Applying KNN imputation for 'CustomerRating' (>20% missing).
2026-07-23 18:56:13,194 - INFO - Detected 46 outliers in 'UnitPrice' using IQR method.
2026-07-23 18:56:13,196 - INFO - Winsorized outliers in 'UnitPrice'.
2026-07-23 18:56:13,203 - INFO - Detected 41 outliers in 'TotalPrice' using IQR method.
2026-07-23 18:56:13,206 - INFO - Winsorized outliers in 'TotalPrice'.


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,ItemsInCart,ReferralSource,CustomerRating,TotalPrice
0,ORD000001,2023-01-01 00:00:00,C25795,Laptop,4.0,206.664611,774 Main St,Online,Shipped,4.0,Referral,3.664780,826.658444
2,ORD000003,2023-01-01 02:00:00,C86820,Desk,3.0,899.067407,704 Main St,Credit Card,Pending,5.0,Instagram,1.813224,2697.202220
3,ORD000004,2023-01-01 03:00:00,C64886,Laptop,2.0,834.992910,141 Main St,Online,Delivered,5.0,Referral,3.103828,1669.985821
4,ORD000005,2023-01-01 04:00:00,C16265,Laptop,4.0,174.746628,895 Main St,Gift Card,Shipped,9.0,Facebook,3.016125,698.986511
5,ORD000006,2023-01-01 05:00:00,C92386,Phone,2.0,132.689025,478 Main St,Online,Shipped,7.0,Instagram,2.597323,265.378051


## 4. Feature Engineering
Creating meaningful features (e.g., IsWeekend, CustomerOrderCount), encoding categoricals, and removing multicollinearity.

In [5]:
engineer = FeatureEngineer(df_clean)
df_featured = engineer.engineer_features()
df_featured = engineer.encode_categorical_features()
df_final = engineer.check_multicollinearity(threshold=0.85)
df_final.head()

2026-07-23 18:56:13,238 - INFO - Engineering new features...
2026-07-23 18:56:13,249 - INFO - Created 'OrderDayOfWeek', 'OrderHour', and 'IsWeekend' features.
2026-07-23 18:56:13,251 - INFO - Created 'AvgItemValueInCart' feature.
2026-07-23 18:56:13,261 - INFO - Created 'CustomerOrderCount' feature.
2026-07-23 18:56:13,263 - INFO - One-Hot Encoding features: ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource']
2026-07-23 18:56:13,275 - INFO - No highly correlated features found.


,OrderID,Date,CustomerID,Quantity,UnitPrice,ShippingAddress,ItemsInCart,CustomerRating,TotalPrice,OrderDayOfWeek,...,PaymentMethod_Gift Card,PaymentMethod_Online,OrderStatus_Delivered,OrderStatus_Pending,OrderStatus_Returned,OrderStatus_Shipped,ReferralSource_Facebook,ReferralSource_Google,ReferralSource_Instagram,ReferralSource_Referral
0,ORD000001,2023-01-01 00:00:00,C25795,4.0,206.664611,774 Main St,4.0,3.664780,826.658444,6,...,False,True,False,False,False,True,False,False,False,True
2,ORD000003,2023-01-01 02:00:00,C86820,3.0,899.067407,704 Main St,5.0,1.813224,2697.202220,6,...,False,False,False,True,False,False,False,False,True,False
3,ORD000004,2023-01-01 03:00:00,C64886,2.0,834.992910,141 Main St,5.0,3.103828,1669.985821,6,...,False,True,True,False,False,False,False,False,False,True
4,ORD000005,2023-01-01 04:00:00,C16265,4.0,174.746628,895 Main St,9.0,3.016125,698.986511,6,...,True,False,False,False,False,True,True,False,False,False
5,ORD000006,2023-01-01 05:00:00,C92386,2.0,132.689025,478 Main St,7.0,2.597323,265.378051,6,...,False,True,False,False,False,True,False,False,True,False


## 5. Final Correlation Heatmap
Checking correlations post feature-engineering.

In [6]:
vis_final = Visualizer(df_final, OUTPUTS_DIR)
vis_final.plot_correlation_heatmap()

2026-07-23 18:56:14,826 - INFO - Correlation heatmap saved.


## 6. Save Processed Data
Saving the final dataset for modeling.

In [7]:
import os
os.makedirs('../data/processed', exist_ok=True)
df_final.to_csv('../data/processed/cleaned_sales.csv', index=False)
print("Data preprocessing complete and saved to 'data/processed/cleaned_sales.csv'")

Data preprocessing complete and saved to 'data/processed/cleaned_sales.csv'
